In [1]:
from ultralytics import YOLO
import cv2
from collections import defaultdict
model = YOLO("yolo26n.pt")
import numpy as np

In [3]:
VIDEO_PATH = "../data/121364_0.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

track_history = defaultdict(list)

while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    result = results[0]

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu()
        track_ids = result.boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):

            x1, y1, x2, y2 = box

            # Bottom-center point of the player box
            x = float((x1 + x2) / 2)
            y = float(y2)

            track_history[track_id].append((x, y))

            # Keep only recent positions
            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

            # Draw bounding box
            cv2.rectangle(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (255, 255, 255),
                2
            )

            # Draw ID
            cv2.putText(
                frame,
                f"ID {track_id}",
                (int(x1), int(y1) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

            # Draw trajectory
            points = track_history[track_id]

            if len(points) > 1:
                pts = np.array(points, dtype=np.int32).reshape((-1, 1, 2))

                cv2.polylines(
                    frame,
                    [pts],
                    isClosed=False,
                    color=(255, 255, 255),
                    thickness=2
                )

            # Draw bottom-center point
            cv2.circle(
                frame,
                (int(x), int(y)),
                4,
                (255, 255, 255),
                -1
            )

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()